# Build AlzKG

Harmonizes two edge sources into one AD-anchored knowledge graph, using the same
edge schema as PrimeKG (`relation`, `display_relation`, `x_id`, `x_type`, `x_name`,
`x_source`, `y_id`, `y_type`, `y_name`, `y_source`), so this graph can be joined
against or diffed with PrimeKG directly. One extra column, `edge_source`, records
which dataset asserted each edge (needed once nodes from several datasets are merged).

1. **AD-specific edges** produced by this repo's `../datasets/processing_scripts/`
   (GWAS Catalog, ClinVar, DisGeNET-AD, ClinicalTrials.gov, PharmGKB, NIAGADS,
   PubTator3 literature mining). Run `../datasets/primary_data_resources.sh` first.
2. **Generic ontology/pathway/interaction edges** already processed by PrimeKG's
   own pipeline (MONDO, HPO, GO, Reactome, ...), read by relative path from
   `../../PrimeKG/datasets/data/` *if present*. Optional: skipped when absent.

Every source names the same entity differently (Alzheimer's disease is
`MONDO:0004975` in GWAS Catalog, `MESH:D000544` in PubTator3, `PA443319` in PharmGKB,
free text in ClinicalTrials.gov; APOE is `348`, `APOE` or `PA55`). Step 3
**harmonizes entities** onto shared identifiers so the sources connect into one graph:

| node type | canonical id | how |
|---|---|---|
| gene/protein | `NCBIGene:<id>` | NCBI `Homo_sapiens.gene_info` symbol/synonym lookup |
| variant | `rs<id>` (dbSNP) | ClinVar RS#, PubTator `RS#:`, PharmGKB/GWAS rsIDs |
| Alzheimer's disease | `MONDO:0004975` | known AD ids + name patterns (incl. stages: mild/probable/...) |
| other disease, drug, phenotype | first id seen for the normalized name | name-based merge across sources |

The final graph is restricted to the AD neighborhood: edges with at least one endpoint
that is an AD node or a direct neighbor of one.

In [1]:
import html
import re
import os
import numpy as np
import pandas as pd

primekg_data = '../../PrimeKG/datasets/data/'
ad_data = '../datasets/data/'
save_path = ad_data + 'kg/'

EDGE_COLS = ['relation', 'display_relation', 'x_id', 'x_type', 'x_name', 'x_source',
             'y_id', 'y_type', 'y_name', 'y_source']

AD_ANCHOR_ID = 'MONDO:0004975'
AD_ANCHOR_NAME = "Alzheimer's disease"

def clean_edges(df):
    cols = EDGE_COLS + (['edge_source'] if 'edge_source' in df.columns else [])
    df = df.get(cols).dropna(subset=['x_id', 'x_type', 'y_id', 'y_type'])
    df = df.drop_duplicates()
    df = df.query('not ((x_id == y_id) and (x_type == y_type))')
    return df

def try_read(path, **kwargs):
    try:
        return pd.read_csv(path, low_memory=False, dtype=str, **kwargs)
    except FileNotFoundError:
        print(f'  [skip] not found: {path}')
        return pd.DataFrame(columns=EDGE_COLS)

## 1. AD-specific edges (this repo's processing_scripts outputs)

In [2]:
# Order matters for node naming: when several sources describe the same node, the
# name from the earliest source is kept (e.g. ClinVar's HGVS names for variants).
sources = [
    ('ClinVar', 'clinvar/clinvar_ad_variant_disease.csv'),
    ('GWAS_Catalog', 'gwas_catalog/gwas_variant_disease.csv'),
    ('GWAS_Catalog', 'gwas_catalog/gwas_gene_disease.csv'),
    ('DisGeNET', 'disgenet/disgenet_ad_gene_disease.csv'),
    ('NIAGADS', 'niagads/niagads_variant_disease.csv'),
    ('NIAGADS', 'niagads/niagads_gene_disease.csv'),
    ('PharmGKB', 'pharmgkb/pharmgkb_ad_edges.csv'),
    ('PubTator3', 'literature/literature_triples.csv'),
    ('ClinicalTrials.gov', 'aact/clinicaltrials_drug_disease.csv'),
]

ad_specific_edges = []
for name, rel_path in sources:
    df = try_read(ad_data + rel_path)
    if len(df):
        df = clean_edges(df).assign(edge_source=name)
        print(f'{name:20s} {rel_path:45s} {len(df):>7,} edges')
        ad_specific_edges.append(df)

ad_specific_edges = pd.concat(ad_specific_edges, ignore_index=True)
print(f'\nTotal AD-specific edges: {len(ad_specific_edges):,}')

ClinVar              clinvar/clinvar_ad_variant_disease.csv          1,620 edges
GWAS_Catalog         gwas_catalog/gwas_variant_disease.csv           8,682 edges
GWAS_Catalog         gwas_catalog/gwas_gene_disease.csv              6,133 edges
  [skip] not found: ../datasets/data/disgenet/disgenet_ad_gene_disease.csv
  [skip] not found: ../datasets/data/niagads/niagads_variant_disease.csv
  [skip] not found: ../datasets/data/niagads/niagads_gene_disease.csv
PharmGKB             pharmgkb/pharmgkb_ad_edges.csv                    392 edges


PubTator3            literature/literature_triples.csv               1,250 edges


ClinicalTrials.gov   aact/clinicaltrials_drug_disease.csv           11,538 edges

Total AD-specific edges: 29,615


## 2. Generic ontology/pathway edges from PrimeKG's pipeline (optional)

If `../../PrimeKG/datasets/data/kg/kg.csv` exists (i.e. you ran PrimeKG's own
`build_graph.ipynb`), pull every edge touching an AD MONDO term (incl. MONDO subtypes)
out of it. PrimeKG stores MONDO ids as bare numbers and genes as NCBI Gene ids, which
are converted to this graph's CURIEs during harmonization below.

In [3]:
mondo_terms = try_read(primekg_data + 'mondo/mondo_terms.csv')
mondo_parents = try_read(primekg_data + 'mondo/mondo_parents.csv')
primekg_kg = try_read(primekg_data + 'kg/kg.csv')

# AD MONDO ids (bare numbers, PrimeKG style) = AD itself + any MONDO descendants.
ad_mondo_ids = {AD_ANCHOR_ID.split(':')[1].lstrip('0')}
if len(mondo_terms):
    ad_mondo_ids |= set(mondo_terms.loc[mondo_terms['name'].str.contains('alzheimer', case=False, na=False), 'id'])
if len(mondo_parents):
    frontier = set(ad_mondo_ids)
    while frontier:
        new = set(mondo_parents.loc[mondo_parents['parent'].isin(frontier), 'child']) - ad_mondo_ids
        ad_mondo_ids |= new
        frontier = new

if len(primekg_kg):
    is_ad_x = primekg_kg['x_id'].isin(ad_mondo_ids) & (primekg_kg['x_type'] == 'disease')
    is_ad_y = primekg_kg['y_id'].isin(ad_mondo_ids) & (primekg_kg['y_type'] == 'disease')
    primekg_ad_edges = clean_edges(primekg_kg[is_ad_x | is_ad_y]).assign(edge_source='PrimeKG')
    print(f'{len(primekg_ad_edges):,} PrimeKG edges touch the AD anchor set directly.')
else:
    primekg_ad_edges = pd.DataFrame(columns=EDGE_COLS + ['edge_source'])
    print('PrimeKG kg.csv not found -- continuing with AD-specific edges only. '
          'Run ../../PrimeKG/knowledge_graph/build_graph.ipynb first to include the '
          'generic-ontology neighborhood.')

all_edges = pd.concat([ad_specific_edges, primekg_ad_edges], ignore_index=True)
print(f'{len(all_edges):,} edges before harmonization')

  [skip] not found: ../../PrimeKG/datasets/data/mondo/mondo_terms.csv
  [skip] not found: ../../PrimeKG/datasets/data/mondo/mondo_parents.csv
  [skip] not found: ../../PrimeKG/datasets/data/kg/kg.csv
PrimeKG kg.csv not found -- continuing with AD-specific edges only. Run ../../PrimeKG/knowledge_graph/build_graph.ipynb first to include the generic-ontology neighborhood.
29,615 edges before harmonization


## 3. Entity harmonization

Map every node onto a canonical `(id, name, source)` so the same gene, variant, drug
or disease coming from different datasets becomes one node.

In [4]:
# --- Genes: NCBI Gene (Homo sapiens) ---------------------------------------------
gene_info = pd.read_csv(ad_data + 'ncbi_gene/Homo_sapiens.gene_info.gz', sep='\t',
                        usecols=['GeneID', 'Symbol', 'Synonyms'], dtype=str)
id_to_symbol = dict(zip(gene_info['GeneID'], gene_info['Symbol']))
symbol_to_id = {s.upper(): g for g, s in zip(gene_info['GeneID'], gene_info['Symbol'])}
# Synonyms are used only when unambiguous and not shadowing an official symbol.
syn = gene_info.assign(Synonyms=gene_info['Synonyms'].str.split('|')).explode('Synonyms')
syn = syn[syn['Synonyms'].notna() & (syn['Synonyms'] != '-')]
syn['key'] = syn['Synonyms'].str.upper()
syn = syn[~syn['key'].isin(symbol_to_id)].drop_duplicates('key', keep=False)
synonym_to_id = dict(zip(syn['key'], syn['GeneID']))
print(f'{len(symbol_to_id):,} official symbols, {len(synonym_to_id):,} unambiguous synonyms')

def harmonize_gene(node_id, name, source):
    node_id, name = str(node_id), str(name)
    if source in ('NCBI', 'NCBIGene'):          # PrimeKG
        node_id = node_id.split(':')[-1]
    first = node_id.split(';')[0]
    if first.isdigit():                         # PubTator3 / DisGeNET / PrimeKG NCBI ids
        gid = first
    else:                                       # symbols (GWAS, ClinVar, NIAGADS) or PharmGKB PA ids
        key = name.upper()
        gid = symbol_to_id.get(key) or synonym_to_id.get(key)
    if gid:
        return f'NCBIGene:{gid}', id_to_symbol.get(gid, name), 'NCBIGene'
    return node_id, name, source

# --- Variants: dbSNP rsIDs --------------------------------------------------------
RS_PATTERN = re.compile(r'(?:RS#:|^rs)(\d+)', flags=re.IGNORECASE)

def harmonize_variant(node_id, name, source):
    node_id, name = str(node_id), str(name)
    m = RS_PATTERN.search(node_id) or RS_PATTERN.search(name)
    if m:
        rsid = f'rs{m.group(1)}'
        return rsid, (name if source == 'ClinVar' else rsid), 'dbSNP'
    if name.startswith('#'):                    # PubTator3 tmVar mention without an rsID
        hgvs = re.search(r'HGVS:([^;]+)', node_id)
        name = hgvs.group(1) if hgvs else node_id.split('tmVar:')[-1]
    return node_id, name, source

# --- Alzheimer's disease anchor ---------------------------------------------------
AD_ANCHOR_IDS = {AD_ANCHOR_ID, 'EFO:0000249', 'MESH:D000544', 'C0002395', 'MedGen:C0002395', 'PA443319'}
# Normalized names that mean "Alzheimer's disease" itself (incl. clinical stage qualifiers),
# e.g. "Alzheimer Disease", "Probable Alzheimer's Disease", "Dementia, Alzheimer Type",
# "AD - Alzheimer's Disease", "Alzheimer's Dementia (AD)".
AD_ANCHOR_NAME_PATTERN = re.compile(
    r'^(?:ad )?(?:(?:probable|possible|mild|moderate|severe|mild to moderate|moderate to severe|early|'
    r'prodromal|sporadic)\s+)*(?:dementia (?:of the )?)?alzheimers?(?: type)?(?:\s+(?:disease|dementia))*'
    r'(?: type)?(?:\s+ad)?$')

def norm_name(name):
    s = html.unescape(str(name)).lower().replace('’', "'").replace("'s", 's').replace("'", '')
    s = re.sub(r'[^a-z0-9]+', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()

def is_ad_anchor(node_id, name, source):
    if source == 'MONDO' and str(node_id).lstrip('0') == '4975':   # PrimeKG bare MONDO id
        return True
    return str(node_id) in AD_ANCHOR_IDS or bool(AD_ANCHOR_NAME_PATTERN.match(norm_name(name)))

def harmonize_disease(node_id, name, source):
    if is_ad_anchor(node_id, name, source):
        return AD_ANCHOR_ID, AD_ANCHOR_NAME, 'MONDO'
    if source in ('MONDO', 'MONDO_grouped'):    # PrimeKG bare MONDO ids
        return f'MONDO:{str(node_id).split("_")[0].zfill(7)}', name, 'MONDO'
    return str(node_id), name, source

HARMONIZERS = {'gene/protein': harmonize_gene, 'variant': harmonize_variant,
               'disease': harmonize_disease}

def harmonize_side(df, side):
    f = df[[f'{side}_id', f'{side}_type', f'{side}_name', f'{side}_source']].drop_duplicates()
    mapped = []
    for node_id, node_type, name, source in f.itertuples(index=False):
        h = HARMONIZERS.get(node_type)
        mapped.append(h(node_id, name, source) if h else (node_id, name, source))
    f[[f'new_{side}_id', f'new_{side}_name', f'new_{side}_source']] = mapped
    df = df.merge(f, on=[f'{side}_id', f'{side}_type', f'{side}_name', f'{side}_source'], how='left')
    df[f'{side}_id'] = df.pop(f'new_{side}_id')
    df[f'{side}_name'] = df.pop(f'new_{side}_name')
    df[f'{side}_source'] = df.pop(f'new_{side}_source')
    return df

kg = harmonize_side(harmonize_side(all_edges, 'x'), 'y')

193,752 official symbols, 65,003 unambiguous synonyms


In [5]:
# --- Name-based merge for everything else (drugs, non-AD diseases, phenotypes) -------
# Nodes of the same type whose normalized names match become one node; the id from the
# highest-priority source wins (curated vocabularies before free-text trial registries).
SOURCE_PRIORITY = ['MONDO', 'NCBIGene', 'dbSNP', 'PrimeKG', 'DrugBank', 'PharmGKB',
                   'PubTator3', 'GWAS_Catalog', 'ClinVar', 'DisGeNET', 'ClinicalTrials.gov']
rank = {s: i for i, s in enumerate(SOURCE_PRIORITY)}

nodes_long = pd.concat([
    kg[['x_id', 'x_type', 'x_name', 'x_source']].set_axis(['id', 'type', 'name', 'source'], axis=1),
    kg[['y_id', 'y_type', 'y_name', 'y_source']].set_axis(['id', 'type', 'name', 'source'], axis=1),
]).drop_duplicates()
merge_types = {'drug/chemical/compound', 'clinical_intervention', 'disease', 'effect/phenotype'}
cand = nodes_long[nodes_long['type'].isin(merge_types)].copy()
cand['key'] = cand['name'].map(norm_name)
cand['rank'] = cand['source'].map(rank).fillna(len(rank))
# A trial intervention that matches a known drug name is that drug.
cand['merge_type'] = cand['type'].replace({'clinical_intervention': 'drug/chemical/compound'})
canon = (cand.sort_values('rank').drop_duplicates(['merge_type', 'key'])
         .set_index(['merge_type', 'key'])[['id', 'type', 'name', 'source']])
canon_map = {}
for r in cand.itertuples(index=False):
    c = canon.loc[(r.merge_type, r.key)]
    if (c['id'], c['type']) != (r.id, r.type):
        canon_map[(r.id, r.type)] = (c['id'], c['type'], c['name'], c['source'])
print(f'{len(canon_map):,} drug/disease/phenotype nodes merged into an equivalent node by name')

for side in ['x', 'y']:
    keys = list(zip(kg[f'{side}_id'], kg[f'{side}_type']))
    hits = [canon_map.get(k) for k in keys]
    m = np.array([h is not None for h in hits])
    if m.any():
        kg.loc[m, [f'{side}_id', f'{side}_type', f'{side}_name', f'{side}_source']] = [h for h in hits if h]

426 drug/disease/phenotype nodes merged into an equivalent node by name


In [6]:
# --- Link AD subtypes / AD-specific traits to the AD anchor -----------------------
# e.g. "Alzheimer disease 3" (PSEN1 familial AD), "late-onset Alzheimers disease",
# "family history of Alzheimer's disease", "Alzheimer's disease biomarker measurement".
nodes_long = pd.concat([
    kg[['x_id', 'x_type', 'x_name', 'x_source']].set_axis(['id', 'type', 'name', 'source'], axis=1),
    kg[['y_id', 'y_type', 'y_name', 'y_source']].set_axis(['id', 'type', 'name', 'source'], axis=1),
]).drop_duplicates(['id', 'type'])
ad_related = nodes_long[nodes_long['type'].isin(['disease', 'effect/phenotype'])
                        & nodes_long['name'].str.contains('alzheimer', case=False, na=False)
                        & (nodes_long['id'] != AD_ANCHOR_ID)]
subtype_edges = pd.DataFrame({
    'relation': np.where(ad_related['type'] == 'disease', 'disease_disease', 'phenotype_disease'),
    'display_relation': np.where(ad_related['type'] == 'disease', 'subtype of', 'related to'),
    'x_id': ad_related['id'], 'x_type': ad_related['type'],
    'x_name': ad_related['name'], 'x_source': ad_related['source'],
    'y_id': AD_ANCHOR_ID, 'y_type': 'disease', 'y_name': AD_ANCHOR_NAME, 'y_source': 'MONDO',
    'edge_source': 'AlzKG_harmonization',
})
print(f'{len(subtype_edges)} AD subtype/trait nodes linked to {AD_ANCHOR_NAME} ({AD_ANCHOR_ID})')
kg = clean_edges(pd.concat([kg, subtype_edges], ignore_index=True))
print(f'{len(kg):,} harmonized edges')

152 AD subtype/trait nodes linked to Alzheimer's disease (MONDO:0004975)


29,087 harmonized edges


## 4. Restrict to the AD neighborhood

Keep edges with at least one endpoint in the AD set (the anchor + its subtypes/traits)
or directly adjacent to it. This drops literature triples that merely co-occur in an AD
paper (e.g. *Retinitis Pigmentosa - MED25*) but keeps the second hop that matters for
repurposing, such as *APOE -> drug* pharmacogenomic edges.

In [7]:
ad_set = {AD_ANCHOR_ID} | set(ad_related['id'])
node_key = lambda side: kg[f'{side}_id'] + '|' + kg[f'{side}_type']
xk, yk = node_key('x'), node_key('y')
ad_keys = {f'{i}|disease' for i in ad_set} | {f'{i}|effect/phenotype' for i in ad_set}
touches_ad = xk.isin(ad_keys) | yk.isin(ad_keys)
hop1 = set(xk[touches_ad]) | set(yk[touches_ad])
before = len(kg)
kg = kg[xk.isin(hop1) | yk.isin(hop1)].copy()
print(f'Kept {len(kg):,} of {before:,} edges within 2 hops of AD '
      f'({touches_ad.sum():,} touch an AD node directly).')

Kept 27,582 of 29,087 edges within 2 hops of AD (20,086 touch an AD node directly).


## 5. Assign node indices and save `ad_kg.csv`

In [8]:
# Node table: one row per (id, type); the first name/source seen wins.
x_nodes = kg[['x_id', 'x_type', 'x_name', 'x_source']].set_axis(
    ['node_id', 'node_type', 'node_name', 'node_source'], axis=1)
y_nodes = kg[['y_id', 'y_type', 'y_name', 'y_source']].set_axis(
    ['node_id', 'node_type', 'node_name', 'node_source'], axis=1)
nodes = pd.concat([x_nodes, y_nodes], ignore_index=True).drop_duplicates(subset=['node_id', 'node_type'])
nodes = nodes.reset_index(drop=True)
nodes.insert(0, 'node_index', nodes.index)

idx = {(r.node_id, r.node_type): r.node_index for r in nodes.itertuples()}
kg['x_index'] = [idx[k] for k in zip(kg['x_id'], kg['x_type'])]
kg['y_index'] = [idx[k] for k in zip(kg['y_id'], kg['y_type'])]
# Use the node table's canonical name everywhere.
name_of = dict(zip(nodes['node_index'], nodes['node_name']))
kg['x_name'] = kg['x_index'].map(name_of)
kg['y_name'] = kg['y_index'].map(name_of)

ad_kg = (kg[['relation', 'display_relation', 'x_index', 'x_id', 'x_type', 'x_name', 'x_source',
             'y_index', 'y_id', 'y_type', 'y_name', 'y_source', 'edge_source']]
         .drop_duplicates(['relation', 'display_relation', 'x_index', 'y_index', 'edge_source'])
         .reset_index(drop=True))

os.makedirs(save_path, exist_ok=True)
ad_kg.to_csv(save_path + 'ad_kg.csv', index=False)
nodes.to_csv(save_path + 'ad_kg_nodes.csv', index=False)
print(f'Wrote {len(ad_kg):,} edges -> {save_path}ad_kg.csv')
print(f'Wrote {len(nodes):,} nodes -> {save_path}ad_kg_nodes.csv')
ad_kg.head()

Wrote 27,537 edges -> ../datasets/data/kg/ad_kg.csv
Wrote 15,813 nodes -> ../datasets/data/kg/ad_kg_nodes.csv


,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source,edge_source
0,variant_disease,pathogenic/likely pathogenic,0,rs119468010,variant,NM_000250.2(MPO):c.1705C>T (p.Arg569Trp),dbSNP,14175,MONDO:0007088,disease,Alzheimer disease type 1,ClinVar,ClinVar
1,variant_disease,conflicting classifications of pathogenicity,1,rs78950939,variant,NM_000250.2(MPO):c.518A>G (p.Tyr173Cys),dbSNP,14175,MONDO:0007088,disease,Alzheimer disease type 1,ClinVar,ClinVar
2,variant_disease,pathogenic/likely pathogenic,2,rs35897051,variant,NM_000250.2(MPO):c.2031-2A>C,dbSNP,14175,MONDO:0007088,disease,Alzheimer disease type 1,ClinVar,ClinVar
3,variant_disease,pathogenic,3,rs63750215,variant,NM_000447.3(PSEN2):c.422A>T (p.Asn141Ile),dbSNP,14176,MONDO:0011743,disease,Alzheimer disease 4,ClinVar,ClinVar
4,variant_disease,pathogenic,4,rs28936379,variant,NM_000447.3(PSEN2):c.715A>G (p.Met239Val),dbSNP,14176,MONDO:0011743,disease,Alzheimer disease 4,ClinVar,ClinVar


## 6. Summary

In [9]:
print('Nodes by type:')
print(nodes['node_type'].value_counts().to_string(), '\n')
print('Edges by relation:')
print(ad_kg['relation'].value_counts().to_string(), '\n')
print('Edges by source dataset:')
print(ad_kg['edge_source'].value_counts().to_string())

Nodes by type:
node_type
variant                   6486
gene/protein              4149
clinical_intervention     2168
drug/chemical/compound    1754
disease                   1254
effect/phenotype             2 

Edges by relation:
relation
variant_disease                    9511
drug_disease                       9404
gene_disease                       6144
literature_association              883
variant_gene                        762
gene_drug                           365
literature_negative_correlation     178
disease_disease                     150
literature_positive_correlation     132
literature_bind                       2
literature_comparison                 2
literature_cotreatment                2
phenotype_disease                     2 

Edges by source dataset:
edge_source
GWAS_Catalog           14815
ClinicalTrials.gov      9404
ClinVar                 1575
PubTator3               1199
PharmGKB                 392
AlzKG_harmonization      152


In [10]:
# How many datasets independently support each gene's link to AD? Genes backed by
# several independent sources are the graph's highest-confidence AD genes.
ad_idx = nodes.loc[nodes['node_id'] == AD_ANCHOR_ID, 'node_index'].iloc[0]
gene_ad = pd.concat([
    ad_kg.query('y_index == @ad_idx and x_type == "gene/protein"')[['x_name', 'edge_source']].set_axis(['gene', 'source'], axis=1),
    ad_kg.query('x_index == @ad_idx and y_type == "gene/protein"')[['y_name', 'edge_source']].set_axis(['gene', 'source'], axis=1),
])
support = (gene_ad.groupby('gene')['source'].agg(lambda s: ', '.join(sorted(set(s))))
           .to_frame('supporting_sources'))
support['n_sources'] = support['supporting_sources'].str.count(',') + 1
support.sort_values(['n_sources', 'gene'], ascending=[False, True]).head(15)

,supporting_sources,n_sources
gene,,
APOE,"GWAS_Catalog, PharmGKB, PubTator3",3
ABCA1,"GWAS_Catalog, PubTator3",2
ABCA7,"GWAS_Catalog, PubTator3",2
ACE,"GWAS_Catalog, PharmGKB",2
ACHE,"PharmGKB, PubTator3",2
ACTN4,"GWAS_Catalog, PubTator3",2
ADAM17,"GWAS_Catalog, PubTator3",2
APP,"GWAS_Catalog, PubTator3",2
BCHE,"PharmGKB, PubTator3",2
